# Module 05 -- Volatility Surface

**Author: Djellal Djouad** -- CrossVol Research | [crossvol.com](https://crossvol.com) | ORCID [0009-0002-4911-1118](https://orcid.org/0009-0002-4911-1118)

The vol surface is the foundation of every derivatives desk. It takes the smiles
from Module 04 and stitches them together across all expirations into a coherent
surface. From this surface, you can price any vanilla option and most exotics.
If your surface is wrong, everything downstream is wrong.

We'll use the SVI (Stochastic Volatility Inspired) parameterization from Gatheral,
which is the industry standard for its parsimony and no-arbitrage properties.

*License: MIT with Educational Use Clause -- see LICENSE. Not trading advice.*


In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm
import matplotlib.pyplot as plt


## SVI Parameterization (Gatheral)

SVI models total implied variance $w(k) = \sigma^2 T$ as a function of
log-moneyness $k = \ln(K/F)$:

$$w(k) = a + b \left( \rho (k - m) + \sqrt{(k - m)^2 + s^2} \right)$$

Five parameters: $a, b, \rho, m, s$. That's it. Five numbers describe an
entire smile slice. The elegance is remarkable.

- $a$: overall variance level
- $b$: slope of the wings
- $\rho$: skew (negative = downside steeper)
- $m$: translation (shifts the minimum)
- $s$: smoothness at the vertex


In [ ]:
def svi_total_variance(k, a, b, rho, m, s):
    """SVI total variance w(k) = sigma^2 * T."""
    return a + b * (rho * (k - m) + np.sqrt((k - m)**2 + s**2))

def svi_implied_vol(k, T, a, b, rho, m, s):
    """Convert SVI total variance to implied vol."""
    w = svi_total_variance(k, a, b, rho, m, s)
    # Guard against negative variance (bad fit)
    w = np.maximum(w, 1e-8)
    return np.sqrt(w / T)


## Fitting SVI to a Smile Slice

We generate synthetic smile data (as if from market quotes) and fit SVI.
In production, you'd feed in actual bid-ask midpoints for each strike.


In [ ]:
def synthetic_market_vols(K, S, _T, atm_vol=0.20, skew_coeff=-0.10, kurt_coeff=0.03):
    """Generate realistic smile data."""
    m = np.log(K / S)
    return atm_vol + skew_coeff * m + kurt_coeff * m**2

S = 100
K_strikes = np.linspace(85, 115, 21)
T_fit = 0.25

market_vols = synthetic_market_vols(K_strikes, S, T_fit)
market_total_var = market_vols**2 * T_fit
log_moneyness = np.log(K_strikes / S)

def svi_objective(params, k_data, w_data):
    a, b, rho, m, s = params
    w_model = svi_total_variance(k_data, a, b, rho, m, s)
    return np.sum((w_model - w_data)**2)

# Initial guess and bounds
x0 = [0.01, 0.1, -0.3, 0.0, 0.1]
bounds = [(-0.5, 0.5), (0.001, 2.0), (-0.999, 0.999), (-0.5, 0.5), (0.001, 2.0)]

result = minimize(svi_objective, x0, args=(log_moneyness, market_total_var),
                  method='L-BFGS-B', bounds=bounds)

a_fit, b_fit, rho_fit, m_fit, s_fit = result.x
print(f"SVI parameters: a={a_fit:.4f}, b={b_fit:.4f}, rho={rho_fit:.4f}, "
      f"m={m_fit:.4f}, s={s_fit:.4f}")
print(f"Fit residual: {result.fun:.2e}")


In [ ]:
# Plot the fit
k_fine = np.linspace(-0.20, 0.15, 200)
fitted_vols = svi_implied_vol(k_fine, T_fit, *result.x)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(log_moneyness, market_vols * 100, 'o', label='Market', markersize=6, color='steelblue')
ax.plot(k_fine, fitted_vols * 100, '-', label='SVI Fit', lw=2, color='firebrick')
ax.set_xlabel('Log-Moneyness ln(K/S)')
ax.set_ylabel('Implied Vol (%)')
ax.set_title(f'SVI Fit -- T={T_fit}Y')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Building the Full Surface

Fit SVI separately for each expiration, then interpolate across tenors.
Here we fit 5 slices and stack them into a surface.


In [ ]:
tenors = [0.04, 0.125, 0.25, 0.5, 1.0]
tenor_labels = ['2W', '6W', '3M', '6M', '1Y']

svi_params_by_tenor = {}

for T_i in tenors:
    # Realistic: ATM vol higher for short-dated, skew steeper for short-dated
    atm = 0.18 + 0.04 * np.exp(-2 * T_i)
    skew = -0.10 / np.sqrt(T_i / 0.25)
    kurt = 0.025 / np.sqrt(T_i / 0.25)

    mvols = synthetic_market_vols(K_strikes, S, T_i, atm_vol=atm,
                                   skew_coeff=skew, kurt_coeff=kurt)
    mvar = mvols**2 * T_i

    res = minimize(svi_objective, x0, args=(log_moneyness, mvar),
                   method='L-BFGS-B', bounds=bounds)
    svi_params_by_tenor[T_i] = res.x

print("Fitted SVI params per tenor:")
for T_i, label in zip(tenors, tenor_labels):
    p = svi_params_by_tenor[T_i]
    print(f"  {label}: a={p[0]:.4f}, b={p[1]:.4f}, rho={p[2]:.4f}, m={p[3]:.4f}, s={p[4]:.4f}")


In [ ]:
# 3D Surface plot
k_surface = np.linspace(-0.18, 0.14, 80)
T_surface = np.linspace(0.04, 1.0, 80)
K_mesh, T_mesh = np.meshgrid(k_surface, T_surface)

# Interpolate SVI params linearly across tenors for the full surface
from scipy.interpolate import interp1d

tenor_arr = np.array(tenors)
param_matrix = np.array([svi_params_by_tenor[t] for t in tenors])

param_interp = interp1d(tenor_arr, param_matrix, axis=0, kind='linear',
                        fill_value='extrapolate')  # type: ignore[arg-type]

IV_surface = np.zeros_like(K_mesh)
for i in range(K_mesh.shape[0]):
    T_val = T_mesh[i, 0]
    params = param_interp(T_val)
    IV_surface[i, :] = svi_implied_vol(K_mesh[i, :], T_val, *params) * 100

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(K_mesh, T_mesh, IV_surface, cmap='plasma', alpha=0.85)
ax.set_xlabel('Log-Moneyness')
ax.set_ylabel('Time to Expiry (Y)')
ax.set_zlabel('Implied Vol (%)')
ax.set_title('Implied Volatility Surface (SVI)')
ax.view_init(elev=25, azim=-45)
fig.colorbar(surf, shrink=0.5, label='IV (%)')
plt.tight_layout()
plt.show()


## Arbitrage Checks

A vol surface that admits arbitrage is useless -- you'd be giving away free money
(or your model says you can make money from nothing, which means the model is wrong).
Two key checks:

### 1. Calendar Spread Arbitrage
Total variance must be non-decreasing in time for every strike:
$w(k, T_1) \leq w(k, T_2)$ when $T_1 < T_2$.

### 2. Butterfly Arbitrage
The probability density implied by the smile must be non-negative. Equivalent to
checking that $\partial^2 C / \partial K^2 \geq 0$ (call prices are convex in strike).


In [ ]:
# Calendar spread check: total variance should increase with tenor
print("Calendar Spread Arbitrage Check:")
print("-" * 50)

k_test_points = np.linspace(-0.15, 0.12, 10)

all_clear = True
for k_val in k_test_points:
    prev_w = 0
    for T_i in tenors:
        p = svi_params_by_tenor[T_i]
        w = svi_total_variance(k_val, *p)
        if w < prev_w - 1e-10:
            print(f"  VIOLATION at k={k_val:.3f}: w(T={T_i:.3f})={w:.6f} < w(prev)={prev_w:.6f}")
            all_clear = False
        prev_w = w

if all_clear:
    print("  No calendar arbitrage detected. Surface is clean.")


In [ ]:
# Butterfly check: second derivative of call price w.r.t. strike must be >= 0
# Equivalent to checking implied density is non-negative
print("\nButterfly Arbitrage Check (3M slice):")
print("-" * 50)

T_check = 0.25
p_check = svi_params_by_tenor[T_check]
k_fine_check = np.linspace(-0.18, 0.14, 500)
w_check = svi_total_variance(k_fine_check, *p_check)
vol_check = np.sqrt(np.maximum(w_check / T_check, 1e-10))

# Price calls using BSM, then check convexity numerically
prices = np.array([
    S * norm.cdf((- k + (0.05 + 0.5 * v**2) * T_check) / (v * np.sqrt(T_check)))
    - S * np.exp(k) * np.exp(-0.05 * T_check) * norm.cdf(
        (-k + (0.05 - 0.5 * v**2) * T_check) / (v * np.sqrt(T_check)))
    for k, v in zip(k_fine_check, vol_check)
])

# Numerical second derivative
dk = k_fine_check[1] - k_fine_check[0]
d2C_dK2 = np.diff(prices, 2) / dk**2

if np.all(d2C_dK2 >= -1e-6):
    print("  No butterfly arbitrage detected. Density is non-negative.")
else:
    n_violations = np.sum(d2C_dK2 < -1e-6)
    print(f"  {n_violations} butterfly violations found (check wing fits).")


## Desk Perspective

The vol surface is something you stare at every single day. After a while you
develop an intuition for what "looks right" vs. what's broken. A few things
I've learned:

- **Short-dated skew is noisy.** The 1-week smile moves around a lot because
  there's less data and events dominate. Don't over-fit it.

- **The long end anchors everything.** If your 1-year vol is wrong, your entire
  surface is wrong. Get the back end right first.

- **SVI is great but not perfect.** For very short-dated or very far OTM options,
  you sometimes need ad-hoc adjustments. Some desks use SSVI (Surface SVI) which
  enforces no-arbitrage globally. Gatheral's later work covers this.

- **Every morning, the first thing you do** is check that the surface updated
  cleanly. If there's a glitch in the feed and one slice is stale, your entire
  book PnL will be off.

---

**Next:** [Module 06 -- Monte Carlo Pricing](06_monte_carlo.py)

*Djellal Djouad -- CrossVol Research -- 2026*
